# Mastra + MongoDB Atlas: 1. Setup, Voyage Embeddings & Vector Search

This is the first of four notebooks building a movie-discovery agent on the `sample_mflix`
dataset with [Mastra](https://mastra.ai), the TypeScript agent framework. They run on a
free Atlas M0 cluster.

The code is TypeScript, executed by the [tslab](https://github.com/yunabe/tslab) kernel.
That is a deliberate trade: Mastra is a TypeScript framework, and a cookbook that showed
it through a Python shim would teach the shim. The cost is that these notebooks do not
run in Colab: they need a local clone and `npm install`.

## What you'll learn
1. Connect `MongoDBVector` to Atlas and confirm the connection
2. Generate Voyage AI embeddings three ways: MongoDB's hosted endpoint, Mastra's
   `voyageEmbedding` provider, and contextualized chunk embeddings
3. Query the plot vectors that ship with `sample_mflix` using `$vectorSearch`
4. Push a metadata filter down into the vector search stage rather than filtering after

## Prerequisites
- A MongoDB Atlas cluster (M0 is fine) with the **`sample_mflix` sample dataset loaded**
- An Atlas API key for the hosted embedding service at `ai.mongodb.com`
- Node.js 20+ and `npm install` run in `partners/mastra/`
- A `.env` at the repository root: copy `.env.example` and fill it in

The notebook set in order:

| Notebook | Covers |
| --- | --- |
| **01 (this one)** | Setup, Voyage embeddings, vector search |
| 02 | Bring-your-own operational collections, read-only by default |
| 03 | Full-text search, hybrid `$rankFusion`, weights, reranking |
| 04 | An agent with retrieval tools and persistent memory |

## Setup

Unlike the Python notebooks in this repository, there is no `%pip install` cell. The
kernel resolves imports from `partners/mastra/node_modules`, so dependencies come from
`npm install` before launching Jupyter:

```bash
cd partners/mastra
npm install
npx tslab install --python="$(which python3)"
```

Two helper modules sit next to this notebook and are shared by all four:

- `src/config.ts`: reads the repository `.env` and applies documented defaults
- `src/utils.ts`: the Atlas embedding and rerank endpoints, plus Claude model setup

Everything else (retrievers, comparisons, agents) stays in the notebooks, where you
can read it.

In [1]:
import { loadNotebookConfig } from './src/config.js';

// Reads the repository .env. A variable the file omits falls back to a documented
// default rather than to whatever your shell happens to hold -- an inherited model id
// from an unrelated project is a genuinely confusing failure.
const cfg = loadNotebookConfig();

console.log('Configuration:', cfg.summary);

Configuration: db=sample_mflix · embeddings=voyage-3-large (2048-d) · llm=claude-haiku-4-5 via gateway


The summary line deliberately carries no secrets. These notebooks commit their outputs so
you can read real results without a cluster of your own, which means every line printed
here ends up in a public repository. `cfg.summary` is built to be safe to print; the
connection string and API keys are not in it.

## 1. Connect to Atlas

`MongoDBVector` is Mastra's MongoDB vector store. Two constructor arguments matter more
than the rest:

- **`embeddingFieldPath`**: which field holds the vectors. The default is `embedding`,
  but we are querying a collection that already exists with its own schema, so we point
  at `plot_embedding_voyage_3_large`.
- **`options.appName`**: a telemetry tag, so MongoDB can attribute this workload.

In [2]:
import { MongoDBVector } from '@mastra/mongodb';
import { APP_NAME } from './src/utils.js';

const vector = new MongoDBVector({
  id: 'mflix',
  uri: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
  // sample_mflix's own field name, not Mastra's `embedding` default.
  embeddingFieldPath: 'plot_embedding_voyage_3_large',
});

await vector.connect();
console.log('Connected to', cfg.dbName);

Connected to sample_mflix


`MongoDBVector` speaks in vectors and indexes. To look at the documents themselves we
use the MongoDB driver directly. The two coexist happily on the same cluster, and
mixing them is normal in a real application.

In [3]:
import { connect } from './src/utils.js';

const { client } = await connect(cfg.mongodbUri);
const movies = client.db(cfg.dbName).collection('embedded_movies');

console.log('embedded_movies documents:', await movies.estimatedDocumentCount());

const sample = await movies.findOne(
  { plot_embedding_voyage_3_large: { $exists: true } },
  { projection: { title: 1, plot: 1, genres: 1, year: 1 } },
);
console.log('\nTitle :', sample?.title);
console.log('Year  :', sample?.year);
console.log('Genres:', sample?.genres);
console.log('Plot  :', String(sample?.plot).slice(0, 140), '...');

embedded_movies documents: 3483



Title : Scarface


Year  : 1932


Genres: [ 'Action', 'Crime', 'Drama' ]


Plot  : An ambitious and near insanely violent gangster climbs the ladder of success in the mob, but his weaknesses prove to be his downfall. ...


Each movie carries its title, plot, genres, cast, and, importantly, a **pre-computed
plot embedding**. `sample_mflix.embedded_movies` ships two vector fields; we use
`plot_embedding_voyage_3_large`, produced by Voyage AI's `voyage-3-large` model at 2048
dimensions.

Because the vectors are already there, we can search immediately. But first we need to
embed *queries* into the same vector space.

## 2. Voyage embeddings, three ways

Vector search compares a query vector against stored vectors. They only line up if both
come from the same model at the same dimension. `voyage-3-large` supports 256, 512, 1024,
and 2048. It **defaults to 1024**. The stored plots are 2048.

A 1024-d query against a 2048-d index is where this gets unpleasant: on a mismatched
`numDimensions` Atlas raises an error, but nothing stops you from *embedding* at the
wrong width and pointing at a differently-built index. So every embed call below sends
the dimension explicitly and checks what came back.

### 2a. MongoDB's hosted Voyage endpoint

Atlas hosts Voyage models at `ai.mongodb.com`, authenticated with your Atlas API key.
No separate Voyage account is needed. `createAtlasEmbedder` in `src/utils.ts` wraps it
and validates the dimension on every call.

In [4]:
import { createAtlasEmbedder } from './src/utils.js';

const embedder = createAtlasEmbedder({
  apiKey: cfg.atlasApiKey,
  model: cfg.embeddingModel,
  dimensions: cfg.embeddingDimensions,
});

const queryVector = await embedder.embedQuery('a criminal mastermind planning a heist');

console.log(`${embedder.model} returned ${queryVector.length} dimensions`);
console.log('First 4 values:', queryVector.slice(0, 4).map((n) => n.toFixed(5)).join(', '));

voyage-3-large returned 2048 dimensions


First 4 values: -0.00474, 0.01523, -0.00650, -0.03004


### 2b. Mastra's `voyageEmbedding` provider

`@mastra/voyageai` ships Voyage as a first-class Mastra embedding provider. Point its
`baseUrl` at `ai.mongodb.com/v1` and it uses the same hosted service with the same Atlas
key, but returns an `EmbeddingModelV3`, the interface `Memory` and the rest of Mastra
accept. This is the form to reach for when the embedder feeds a Mastra component rather
than your own code.

Note `inputType: 'query'`. Voyage prepends a different instruction for queries than for
documents, which measurably improves retrieval. `createAtlasEmbedder` above picks it per
call; here it is fixed at construction.

In [5]:
import { voyageEmbedding } from '@mastra/voyageai';

const voyageQueryModel = voyageEmbedding({
  model: 'voyage-3-large',
  apiKey: cfg.atlasApiKey,
  baseUrl: 'https://ai.mongodb.com/v1',
  inputType: 'query',
  outputDimension: 2048,
});

const viaMastra = await voyageQueryModel.doEmbed({
  values: ['a criminal mastermind planning a heist'],
});

console.log('Mastra provider dimensions:', viaMastra.embeddings[0].length);
console.log('First 4 values:', viaMastra.embeddings[0].slice(0, 4).map((n) => n.toFixed(5)).join(', '));

// Same model, same service, same input -> the two paths agree.
const maxDelta = Math.max(...queryVector.map((v, i) => Math.abs(v - viaMastra.embeddings[0][i])));
console.log('\nLargest difference vs. the direct call:', maxDelta.toExponential(2));

Mastra provider dimensions: 2048


First 4 values: -0.00474, 0.01523, -0.00650, -0.03004



Largest difference vs. the direct call: 0.00e+0


### 2c. Contextualized chunk embeddings

`voyage-context-3` embeds chunks *with knowledge of the surrounding document*. A chunk
reading "he was arrested that night" embeds differently depending on what came before it,
which helps when you have split long documents and the chunks alone are ambiguous.

This does not apply to `sample_mflix`: a movie plot is one short chunk. It is here
because it is the right tool for the long-document RAG case that follows a cookbook like
this one, and because the input shape is nested one level deeper than you would guess:
an array of documents, each an array of chunks.

In [6]:
import { voyageContextualizedEmbedding } from '@mastra/voyageai';

const contextModel = voyageContextualizedEmbedding({
  model: 'voyage-context-3',
  apiKey: cfg.atlasApiKey,
  baseUrl: 'https://ai.mongodb.com/v1',
});

// One document, split into three chunks. Note the nesting: values is
// Array<Array<string>> -- documents of chunks, not a flat list of chunks.
const chunked = await contextModel.doEmbed({
  values: [[
    'A team of thieves cases a Monte Carlo casino for six months.',
    'The heist itself takes eleven minutes.',
    'He was arrested that night, two blocks from the vault.',
  ]],
} as any);

console.log('Chunks embedded:', chunked.embeddings.length);
console.log('Dimensions per chunk:', chunked.embeddings[0].length);
console.log('\nEach chunk vector encodes its neighbours, so the third chunk is');
console.log('interpreted as part of a heist story rather than in isolation.');

Chunks embedded: 3


Dimensions per chunk: 1024



Each chunk vector encodes its neighbours, so the third chunk is


interpreted as part of a heist story rather than in isolation.


## 3. Register the vector index

Atlas needs a **vector search index** to run semantic queries. This collection already
has one, `movies_vector_index`, created by the LangChain notebooks in this repository
(or by the Atlas UI). It declares `genres` and `year` as filter fields.

So there is nothing to *provision*. What we do instead is tell Mastra where it is:
`createIndex` with `collectionName` and `searchIndexName` registers an existing
collection and its existing Atlas index as a Mastra logical index. Notebook 02 covers
this bring-your-own path properly, including why it is read-only by default. Here it is
just the registration we need in order to query.

`filterFields` is the load-bearing argument. Declaring `genres` and `year` lets Mastra
push a matching filter *into* the `$vectorSearch` stage. Section 5 shows what that buys.

In [7]:
const INDEX = 'mflix_plots';

await vector.createIndex({
  indexName: INDEX,
  dimension: cfg.embeddingDimensions,
  metric: 'cosine',
  // Point at the existing collection and its existing Atlas index. Because
  // collectionName is set, Mastra never creates or drops this collection.
  collectionName: 'embedded_movies',
  searchIndexName: 'movies_vector_index',
  // Must match the filter fields the Atlas index actually declares.
  filterFields: ['genres', 'year'],
});

console.log('Registered logical index:', INDEX);
console.log('Mastra logical indexes:', await vector.listIndexes());

Registered logical index: mflix_plots


Mastra logical indexes: [ 'mflix_plots' ]


`listIndexes()` returns *logical* index names, not Atlas index names or collection names.
`mflix_plots` is a Mastra-level handle that happens to point at `embedded_movies` and
`movies_vector_index`. Keeping those distinct is what lets one collection back several
logical indexes with different filter fields.

## 4. Semantic search

`query` embeds nothing itself. You hand it a vector. That is the honest division of
labour: the store searches, the embedder embeds, and you can see which model produced
the vector you are searching with.

`metadataMode: 'document'` is what makes this work against an existing collection. The
default `'field'` projects Mastra's own managed `metadata` subdocument, which
`embedded_movies` does not have. `'document'` returns the whole source document instead,
so you get `title`, `genres`, and `year` as they are actually stored.

In [8]:
interface Movie {
  title?: string;
  genres?: string[];
  year?: number;
  plot?: string;
}

const results = await vector.query({
  indexName: INDEX,
  // Written out rather than as the shorthand `queryVector,`: the tslab kernel does not
  // resolve a shorthand property against a variable declared in an earlier cell.
  queryVector: queryVector,       // section 2a: 'a criminal mastermind planning a heist'
  topK: 5,
  metadataMode: 'document',       // return the source document, not a managed subdocument
});

for (const hit of results) {
  const movie = hit.metadata as Movie;
  console.log(`${hit.score.toFixed(4)}  ${String(movie.title).padEnd(32)} ${movie.year}  ${movie.genres?.join(', ')}`);
}

0.7933  Parker                           2013  Action, Crime, Thriller


0.7912  The Hot Rock                     1972  Action, Comedy, Crime


0.7893  The Italian Job                  2003  Action, Crime, Thriller


0.7884  After the Sunset                 2004  Action, Comedy, Crime


0.7880  Tower Heist                      2011  Action, Comedy, Crime


The results are thematically about heists even where the plots never use the word. That
is semantic search: the query vector is near the plot vectors in meaning-space, and
nothing was matched literally.

The scores are cosine similarities mapped into 0 to 1. Their absolute values are not very
interesting; their *order* is the result.

## 5. Filtering: pushdown vs. pre-filter

Restricting a vector search to a subset (one genre, one decade) has two possible
implementations, and the difference is not cosmetic.

**Pushdown.** The filter goes inside the `$vectorSearch` stage. Atlas applies it while
traversing the index, so the ANN search only ever considers matching documents.

**Pre-filter.** The filter runs first as a `$match`, collecting the `_id`s that qualify,
and the vector search is then restricted to `{ _id: { $in: [...] } }`. This accepts any
filter the MongoDB query language supports, but the `_id` list travels inside the
aggregation, so on a large collection it can hit the 16 MB BSON document limit.

Mastra chooses per query. A filter is pushed down when it touches only fields declared in
`filterFields` **and** uses only operators `$vectorSearch.filter` accepts (`$and`, `$or`,
`$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`). Anything else falls back to
the pre-filter. We declared `genres` and `year`, so this one is pushed down.

In [9]:
const fantasy = await vector.query({
  indexName: INDEX,
  queryVector: await embedder.embedQuery('an epic quest to save the kingdom'),
  topK: 5,
  // genres and year are declared filter fields, and $eq/$gte are supported
  // operators -- so this filter is pushed into $vectorSearch rather than
  // pre-filtering _ids.
  filter: { genres: { $eq: 'Fantasy' }, year: { $gte: 1990 } },
  metadataMode: 'document',
});

for (const hit of fantasy) {
  const movie = hit.metadata as Movie;
  console.log(`${hit.score.toFixed(4)}  ${String(movie.title).padEnd(32)} ${movie.year}  ${movie.genres?.join(', ')}`);
}

0.7683  Your Highness                    2011  Adventure, Comedy, Fantasy


0.7643  Mythica: A Quest for Heroes      2014  Action, Adventure, Fantasy


0.7523  The Legend of Galgameth          1996  Adventure, Family, Fantasy


0.7523  The Legend of Galgameth          1996  Adventure, Family, Fantasy


0.7512  The Scorpion King                2002  Action, Adventure, Fantasy


Every result is a Fantasy film from 1990 or later, still ranked by how well its plot
matches the quest description.

If you see the same title twice, that is the dataset rather than the search: `sample_mflix`
contains genuine duplicate movie documents, and two documents with near-identical plots
land next to each other. Deduplicating retrieval results by some business key is a normal
thing to do in production.

Now the other path. `rated` is a real field on these documents but is **not** in our
`filterFields`, so this query cannot be pushed down and Mastra falls back to the
pre-filter. Against this index that fails, and the error is worth reading rather than
avoiding.

In [10]:
try {
  await vector.query({
    indexName: INDEX,
    queryVector: await embedder.embedQuery('an epic quest to save the kingdom'),
    topK: 3,
    // `rated` is not a declared filter field, so Mastra takes the pre-filter path
    // and rewrites this into `$vectorSearch.filter = { _id: { $in: [...] } }`.
    filter: { rated: { $eq: 'PG' } },
    metadataMode: 'document',
  });
} catch (error) {
  console.log('Failed:', (error as Error).message.split('\n')[0]);
}

Failed: PlanExecutor error during aggregation :: caused by :: Path '_id' needs to be indexed as filter


> `Path '_id' needs to be indexed as filter`

The pre-filter is not a filter *outside* the vector search; it becomes
`$vectorSearch.filter = { _id: { $in: [ ...matching ids... ] } }`. And Atlas only accepts
a path inside `$vectorSearch.filter` if the index **declares that path as a filter
field**. `movies_vector_index` declares `genres` and `year`. It does not declare `_id`.
So the fallback has nothing to stand on.

This is a genuinely sharp edge, because the failure is a property of the *index
definition* rather than of your query, and it only appears on the fallback path. Three
ways out, in the order worth trying:

1. **Declare the field you filter on.** Add `rated` as a `filter` field on the Atlas
   index and to `filterFields`. Now it is pushed down and the fallback never runs. Best
   answer when you know your filters in advance.
2. **Declare `_id` as a filter field.** One `{ type: 'filter', path: '_id' }` in the
   index definition makes the pre-filter work for *every* undeclared field. Verified:
   with `_id` declared, the exact query above returns results. This is what to do when
   filters are user-driven and you cannot enumerate the fields, with the 16 MB caveat
   still in force.
3. **Filter after retrieval.** Over-fetch and filter the results in your own code.
   Always available, and it changes what `topK` means.

We take option 1 by construction: `genres` and `year` are declared, and they are what
the later notebooks filter on.

## 6. Retrieval, in a function

The three notebooks that follow all need "text in, movies out". Before moving on, here
is that step as one function, so the later notebooks can reuse the shape rather than
repeating the embed-then-query dance.

In [11]:
/** Embed `text` and return the closest movies. The retrieval half of RAG. */
async function searchMovies(text: string, topK = 5): Promise<Array<Movie & { score: number }>> {
  const vec = await embedder.embedQuery(text);
  const hits = await vector.query({
    indexName: INDEX,
    queryVector: vec,
    topK,
    metadataMode: 'document',
  });
  return hits.map((hit) => ({ ...(hit.metadata as Movie), score: hit.score }));
}

for (const movie of await searchMovies('a lonely robot on an abandoned planet', 3)) {
  console.log(`${movie.score.toFixed(4)}  ${movie.title} (${movie.year})`);
}

0.7894  The Iron Giant (1999)


0.7755  Red Planet (2000)


0.7689  Abiogenesis (2011)


## Cleanup

Two things are worth being careful about here, and both are easy to get wrong.

`deleteIndex` on a bring-your-own registration **drops the Atlas search index the
registration points at**. Our registration points at `movies_vector_index`, which is
shared with the other notebooks in this repository, so calling `deleteIndex` would
destroy it, leave the collection intact, and make every later notebook return zero
results with no obvious cause.

We also leave the registration itself in place, because notebooks 03 and 04 reuse it.

In [12]:
// NOT vector.deleteIndex({ indexName: INDEX }) -- see above. The registration is
// bookkeeping in a Mastra-owned collection; the Atlas index it points at is not ours
// to delete.
await vector.disconnect();
await client.close();

console.log('Disconnected. The `mflix_plots` registration is left in place for');
console.log('notebooks 03 and 04, and movies_vector_index is untouched.');

Disconnected. The `mflix_plots` registration is left in place for


notebooks 03 and 04, and movies_vector_index is untouched.


## Recap

You connected `MongoDBVector` to Atlas, generated Voyage embeddings through MongoDB's
hosted endpoint and through Mastra's own provider, registered the existing
`sample_mflix` vector index as a Mastra logical index, and ran semantic search with
filters taking both the pushdown and pre-filter paths.

Two things to carry forward:

- **Dimensions have to match, and the default is not the answer.** `voyage-3-large`
  defaults to 1024; the sample data is 2048. Set it explicitly and check the result.
- **`metadataMode: 'document'`** is what makes any of this work against a collection
  that already has its own schema.

**Next:** `02_byo_operational_collections.ipynb` looks properly at what we skimmed past
in section 3: registering collections you already own, why writes are refused by
default, and what `allowWrites` actually opts into.